## Importações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

years = [2019, 2020, 2021, 2022, 2023, 2024]

modes = {}
graphs = {}

base_modes = "../../data/03_modes"

for year in years:
    for speed in ["fast", "medium", "slow"]:
        file_path = f"../../data/04_graphs/{year}/{speed}_graph.gpickle"
        with open(file_path, "rb") as f:
            graphs[f"graph_{speed}_{year}"] = pickle.load(f)
                    
            modes[f"{speed}_{year}"] = pd.read_parquet(f"../../data/03_modes/{year}/{speed}_emd.parquet")

## Funções

In [2]:
import community as community_louvain

def calculate_hybrid_risk_metric(G, weights=None):
    """
    Calculate the hybrid risk metric R_i for each node in PMFG graph G.
    
    Parameters:
        G (networkx.Graph): PMFG graph with weighted edges (correlations)
        weights (dict): Optional weights for the metrics:
            {
                'degree': float,
                'clustering': float,
                'betweenness': float,
                'edge_weight': float,
                'community': float,
                'kcore': float
            }
            Defaults to equal weights if None.
    
    Returns:
        dict: {node: risk_score (0-1, lower = less risk)}
    """
    # Default equal weights if not provided
    if weights is None:
        weights = {
            'degree': 1/4,
            'clustering': 1/4,
            'betweenness': 1/4,
            'edge_weight': 1/4,
            'community':0,
            'kcore': 0
        }
    
    nodes = list(G.nodes)
    
    # 1. Calculate raw metrics
    
    # Degree
    degree_dict = dict(G.degree())
    
    # Clustering coefficient
    clustering_dict = nx.clustering(G, weight='weight')
    
    # Betweenness centrality
    betweenness_dict = nx.betweenness_centrality(G, weight='weight', normalized=True)
    
    # Average edge weight per node (mean correlation of edges)
    avg_edge_weight = {}
    for node in nodes:
        edges = G.edges(node, data='weight', default=0)
        weights_list = [abs(data) for _, _, data in edges]  # absolute value of correlations
        avg_edge_weight[node] = np.mean(weights_list) if weights_list else 0
    
    # Community detection (Louvain)
    partition = community_louvain.best_partition(G, weight='weight', random_state=0)
    
    # Count unique communities connected to node (including its own)
    community_of_node = {node: partition[node] for node in nodes}
    community_neighbors_count = {}
    for node in nodes:
        neighbors = G.neighbors(node)
        neighbor_communities = set([partition[n] for n in neighbors])
        # Include own community
        neighbor_communities.add(partition[node])
        community_neighbors_count[node] = len(neighbor_communities)
    
    # K-core number
    kcore_dict = nx.core_number(G)
    
    # 2. Normalize metrics (0=lowest risk, 1=highest risk)
    def normalize_dict(d, invert=False):
        vals = np.array(list(d.values()))
        min_val = vals.min()
        max_val = vals.max()
        if max_val - min_val == 0:
            # Avoid division by zero if all values equal
            return {k: 0.0 for k in d.keys()}
        norm = {k: (v - min_val) / (max_val - min_val) for k, v in d.items()}
        if invert:
            norm = {k: 1 - v for k, v in norm.items()}
        return norm
    
    norm_degree = normalize_dict(degree_dict, invert=False)
    norm_clustering = normalize_dict(clustering_dict, invert=False)
    norm_betweenness = normalize_dict(betweenness_dict, invert=False)
    norm_edge_weight = normalize_dict(avg_edge_weight, invert=False)
    norm_community = normalize_dict(community_neighbors_count, invert=True)  # higher community count = lower risk
    norm_kcore = normalize_dict(kcore_dict, invert=False)
    
    # 3. Combine metrics with weights
    risk_scores = {}
    for node in nodes:
        score = (
            weights['degree'] * norm_degree[node] +
            weights['clustering'] * norm_clustering[node] +
            weights['betweenness'] * norm_betweenness[node] +
            weights['edge_weight'] * norm_edge_weight[node] +
            weights['community'] * norm_community[node] +
            weights['kcore'] * norm_kcore[node]
        )
        risk_scores[node] = score
    
    return risk_scores

## Construção do DataFrame com os nós e as métricas de grafos calculadas para cada modo

In [ ]:
data = []

for name, G in graphs.items():
    _, speed, year = name.split("_")

    hrm = calculate_hybrid_risk_metric(G)

    for node in G.nodes():
        data.append({
            "node": node,
            "hrm": hrm[node],
            "speed": speed.capitalize(),
            "year": year
        })

df = pd.DataFrame(data)

df["rank"] = (
    df.groupby(["year", "speed"])["hrm"]
      .rank(ascending=False, method="first")
)

df["n_graph"] = (
    df.groupby(["year", "speed"])["node"]
      .transform("count")
)

df["ext_central"] = 1 - (df["rank"] - 1) / (df["n_graph"] - 1)
df["ext_peripheral"] = (df["rank"] - 1) / (df["n_graph"] - 1)

records = []

for _, r in df.iterrows():
    for pos, ext in [("Central", r["ext_central"]),
                     ("Peripheral", r["ext_peripheral"])]:
        records.append({
            "year": r["year"],
            "node": r["node"],
            "speed": r["speed"],
            "position": pos,
            "extremeness": ext
        })

df_choices = pd.DataFrame(records)

df_final = (
    df_choices
    .sort_values("extremeness", ascending=False)
    .groupby(["year", "node"], as_index=False)
    .first()
)

df_central = df_choices.query(
    "position == 'Central' and extremeness >= 0.75"
)

central_all_modes = (
    df_central
    .groupby(["year", "node"])["speed"]
    .nunique()
    .reset_index(name="n_speeds")
)

central_all_modes = central_all_modes.query("n_speeds == 3")

In [46]:
df_central = df_choices.query(
    "position == 'Central' and extremeness > 0.75"
)

central_all_modes = (
    df_central
    .groupby(["year", "node"])["speed"]
    .nunique()
    .reset_index(name="n_speeds")
)

central_all_modes = central_all_modes.query("n_speeds == 3")

In [47]:
df_peripheral = df_choices.query(
    "position == 'Peripheral' and extremeness > 0.75"
)

peripheral_all_modes = (
    df_peripheral
    .groupby(["year", "node"])["speed"]
    .nunique()
    .reset_index(name="n_speeds")
)

peripheral_all_modes = peripheral_all_modes.query("n_speeds == 3")

In [48]:
central_2019 = central_all_modes.query("year=='2019'")
central_2020 = central_all_modes.query("year=='2020'")
central_2021 = central_all_modes.query("year=='2021'")
central_2022 = central_all_modes.query("year=='2022'")
central_2023 = central_all_modes.query("year=='2023'")
central_2024 = central_all_modes.query("year=='2024'")

peripheral_2019 = peripheral_all_modes.query("year=='2019'")
peripheral_2020 = peripheral_all_modes.query("year=='2020'")
peripheral_2021 = peripheral_all_modes.query("year=='2021'")
peripheral_2022 = peripheral_all_modes.query("year=='2022'")
peripheral_2023 = peripheral_all_modes.query("year=='2023'")
peripheral_2024 = peripheral_all_modes.query("year=='2024'")

In [49]:
def churn_by_year(df, label):
    years = sorted(df["year"].unique())

    records = []

    for y_prev, y_curr in zip(years[:-1], years[1:]):
        prev_nodes = set(df.query("year == @y_prev")["node"])
        curr_nodes = set(df.query("year == @y_curr")["node"])

        entered = curr_nodes - prev_nodes
        exited = prev_nodes - curr_nodes
        stayed = curr_nodes & prev_nodes

        records.append({
            "group": label,
            "year": y_curr,
            "entered": len(entered),
            "exited": len(exited),
            "stayed": len(stayed),
            "prev_total": len(prev_nodes),
            "curr_total": len(curr_nodes)
        })

    return pd.DataFrame(records)

central_churn = churn_by_year(central_all_modes, "Central")
peripheral_churn = churn_by_year(peripheral_all_modes, "Peripheral")

In [50]:
central_churn

,group,year,entered,exited,stayed,prev_total,curr_total
0,Central,2020,20,18,0,18,20
1,Central,2021,14,20,0,20,14
2,Central,2022,22,14,0,14,22
3,Central,2023,22,22,0,22,22
4,Central,2024,20,18,4,22,24


In [51]:
peripheral_churn

,group,year,entered,exited,stayed,prev_total,curr_total
0,Peripheral,2020,25,42,14,56,39
1,Peripheral,2021,43,27,12,39,55
2,Peripheral,2022,30,38,17,55,47
3,Peripheral,2023,45,31,16,47,61
4,Peripheral,2024,34,26,35,61,69


In [57]:
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]
years = range(2019, 2025)

def get_portfolio_info(
        portfolio
):
    stock_info = pd.read_csv("../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv")
    screener = pd.read_parquet("../../data/01_raw/screener_result.parquet")[["Ticker", "Country"]]
    stock_info = stock_info.merge(
        screener,
        how="left",
        on="Ticker"
    )
    portfolio = portfolio.to_frame().rename(columns={"node":"Ticker"})
    return portfolio.merge(
        stock_info, on="Ticker", how="left"
    )


for year in years:
    central_temp = central_all_modes.query(f"year=='{year}'")["node"]
    peripheral_temp = peripheral_all_modes.query(f"year=='{year}'")["node"]
    returns_temp = pd.read_parquet(
        f"../../data/02_clean/returns_{year}.parquet"
    )
    port_central_temp = returns_temp[central_temp]
    port_peripheral_temp = returns_temp[peripheral_temp]

    metadata_central_temp = get_portfolio_info(central_temp)
    metadata_peripheral_temp = get_portfolio_info(peripheral_temp)

    port_central_temp.to_csv(f"../../data/06_portfolios/central_portfolio_{year}")
    port_peripheral_temp.to_csv(f"../../data/06_portfolios/peripheral_portfolio_{year}")

    metadata_central_temp.to_csv(f"../../data/07_portfolios_metadata/central_portfolio_{year}")
    metadata_peripheral_temp.to_csv(f"../../data/07_portfolios_metadata/peripheral_portfolio_metadata_{year}")

In [56]:
metadata_peripheral_temp

,Ticker,Company,Sector,Industry,mcap_2019,mcap_2020,mcap_2021,mcap_2022,mcap_2023,mcap_2024,...,gpm_2022,gpm_2023,gpm_2024,epsdg_2019,epsdg_2020,epsdg_2021,epsdg_2022,epsdg_2023,epsdg_2024,Country
0,ADP,Automatic Data Processing Inc,Technology,Software - Application,73315000000,75025960000,103095098000,98959698000,95773967000,73315000000,...,"0,468569489","0,4888631039","0,4994427838","0,2329411765","0,08782442748","0,06483983018","0,1537118192","0,1720740275","0,1089938839",USA
1,AGNG,Global X Aging Population ETF,Financial,Exchange Traded Fund,47384508,56153457,57943449,52674035,57199295,47384508,...,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,USA
2,AVBH,Avidbank Holdings Inc,Financial,Banks - Regional,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,...,1,"1,802549179",1,"0,1421052632","-0,2580645161","0,2546583851","0,7821782178","-0,3777777778","0,2321428571",USA
3,BANX,ArrowMark Financial Corp,Financial,Closed-End Fund - Debt,146304948,127324062,152877278,121028747,129472646,146304948,...,1,1,1,"0,2549019608","-0,3854166667","0,593220339","-0,6010638298","2,786666667","-0,1725352113",USA
4,BGRN,iShares USD Green Bond ETF,Financial,Exchange Traded Fund,401617586,421237912,406335451,344254343,353690073,401617586,...,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,USTB,VictoryShares Short-Term Bond ETF,Financial,Exchange Traded Fund,1133808317,1160921124,1150165631,1089890050,1111176965,1133808317,...,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,USA
65,VCSH,Vanguard Short-Term Corporate Bond ETF,Financial,Exchange Traded Fund,42782898836,43955033051,42904336165,39699446668,40850461347,42782898836,...,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,USA
66,VSDA,VictoryShares Dividend Accelerator ETF,Financial,Exchange Traded Fund,175154913,194170052,233229500,218821121,233180492,175154913,...,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,#ERROR!,USA
67,WHF,WhiteHorse Finance Inc,Financial,Asset Management,281480638,279706799,359455168,303322298,285889982,281480638,...,"0,493220613","0,4619313992",1,"-0,4623655914","0,02666666667","-0,07792207792","-0,5211267606","0,2941176471","-0,4659090909",USA
